# Demo: PATCH endpoint in Catalog

Demo for story RSPY-79: https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-79

## 1 - Build Catalog

In [ ]:
import pprint
import pystac
from pystac import ItemCollection
# Init environment before running a demo notebook.
from resources.utils import *

init_demo()

# Reload the global vars again
from resources.utils import *

pp = pprint.PrettyPrinter(indent=2, width=80, sort_dicts=False, compact=True)
cadip_collection_id = "cadip_sentinel1"

# Init the dask cluster
from resources.dask_clusters.dask_main_env import *
await init_dask_cluster_staging()

In [ ]:
# Create a test collection 
collection = create_test_collection()

# Get all the items from the collection "cadip_sentinel1" found in the configuration of the CADIP station
items_collection_cadip = list(cadip_client.get_items(cadip_collection_id))
assert len(items_collection_cadip) > 0

# Starting staging process from the CADIP station
staging_resp = staging_client.run_staging(pystac.ItemCollection(items_collection_cadip).to_dict(), TEST_COLLECTION)
    
staging_client.wait_for_jobs(staging_resp, logger)

In [ ]:
# Check the catalog for my_test_collection
result = list(catalog_client.get_items(TEST_COLLECTION))

for item in result:
    print(f"Item {item.id} has {len(item.assets)} assets")

TEST_ITEM_ID = result[0].id

## 2 - Get initial STAC details

Retrieve STAC description of a collection and an item to see their initial content.

In [ ]:
# STAC description of the collection
initial_collection = catalog_client.get_collection(TEST_COLLECTION)
initial_collection

In [ ]:
# STAC description of an item
initial_item = catalog_client.get_item(TEST_COLLECTION, TEST_ITEM_ID)
initial_item

## 3 - Patch Collection and Item

Run two PATCH requests: one on the collection and one on the item, with one field to change in each case.

In [ ]:
# Patch Collection
patch_values = {
    "description": "Brand New Description"
}

catalog_client.patch_collection(collection_id=TEST_COLLECTION, 
                                owner_id=OWNER_ID,
                                patch_values=patch_values)

In [ ]:
# Patch Item
patch_values = {
    "properties":{
        "constellation":"orion"
    }
}

catalog_client.patch_item(collection_id=TEST_COLLECTION, 
                          item_id=TEST_ITEM_ID,
                          owner_id=OWNER_ID,
                          patch_values=patch_values)

## 4 - Get modified STAC details

Retrieve STAC descriptions of the same elements as in step 2, and check that the fields were properly updated along with the "updated" timestamp.

In [ ]:
# STAC description of the same collection
modified_collection = catalog_client.get_collection(TEST_COLLECTION)

# Check differences between original collection and patched one
assert initial_collection.id == modified_collection.id
assert modified_collection.description == "Brand New Description"
#assert modified_collection.updated > initial_collection.updated

modified_collection

In [ ]:
# STAC description of the same item
modified_item = catalog_client.get_item(TEST_COLLECTION, TEST_ITEM_ID)

# Check differences between original item and patched one
assert initial_item.id == modified_item.id
assert initial_item.properties["constellation"] != "orion"
assert modified_item.properties["constellation"] == "orion"
assert modified_item.properties["updated"] > initial_item.properties["updated"]

modified_item

## 5 - Delete the catalog collection

In [ ]:
result = catalog_client.remove_collection(TEST_COLLECTION)
assert result.json()["deleted collection"] == TEST_COLLECTION
pp.pprint(result.json())